In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

In [3]:
PROJECT_ROOT = Path("..")

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed data directory:")
print(PROCESSED_DATA.resolve())

Processed data directory:
E:\Project\spam-ham-detection\data\processed


In [4]:
DATA_PATH = RAW_DATA / "enron_spam_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (33716, 5)


In [5]:
df = df.rename(columns={
    "Message ID": "message_id",
    "Subject": "subject",
    "Message": "message",
    "Spam/Ham": "label",
    "Date": "date"
})

print(df.columns.tolist())

['message_id', 'subject', 'message', 'label', 'date']


In [6]:
df["subject"] = df["subject"].fillna("")
df["message"] = df["message"].fillna("")

In [7]:
print(df[["subject", "message"]].isna().sum())

subject    0
message    0
dtype: int64


In [8]:
df["text"] = (
    df["subject"].astype(str).str.strip()
    + " "
    + df["message"].astype(str).str.strip()
)

df["text"] = (
    df["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [9]:
df["label"] = (
    df["label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df["target"] = (
    df["label"] == "spam"
).astype(int)

In [10]:
print(df["label"].value_counts())
print()
print(df["target"].value_counts())

label
spam    17171
ham     16545
Name: count, dtype: int64

target
1    17171
0    16545
Name: count, dtype: int64


In [11]:
df["email_content"] = (
    df["subject"].astype(str).str.strip()
    + " "
    + df["message"].astype(str).str.strip()
)

df["email_content"] = (
    df["email_content"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["group_id"] = (
    df["email_content"]
    .astype("category")
    .cat.codes
)

In [12]:
group_label_counts = (
    df.groupby("group_id")["target"]
    .nunique()
)

conflicting_groups = (
    group_label_counts > 1
)

print(
    "Groups with conflicting labels:",
    conflicting_groups.sum()
)

Groups with conflicting labels: 0


In [13]:
X = df["text"]
y = df["target"]
groups = df["group_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

In [14]:
df["split"] = "train"

df.loc[
    df.index.isin(df.index[test_idx]),
    "split"
] = "test"

In [15]:
print(df["split"].value_counts())

split
train    27021
test      6695
Name: count, dtype: int64


In [16]:
train_df = df[
    df["split"] == "train"
].copy()

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.125,
    random_state=42
)

train_sub_idx, val_idx = next(
    gss_val.split(
        train_df["text"],
        train_df["target"],
        groups=train_df["group_id"]
    )
)

In [17]:
validation_indices = train_df.iloc[
    val_idx
].index

df.loc[
    validation_indices,
    "split"
] = "validation"

In [18]:
print(
    df["split"].value_counts()
)

split
train         23663
test           6695
validation     3358
Name: count, dtype: int64


In [19]:
train_df = df[
    df["split"] == "train"
].copy()

val_df = df[
    df["split"] == "validation"
].copy()

test_df = df[
    df["split"] == "test"
].copy()

In [20]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 23663
Validation: 3358
Test: 6695
Total: 33716


In [21]:
for name, data in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name}")
    print(data["label"].value_counts())
    print(
        data["label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


TRAIN
label
spam    12079
ham     11584
Name: count, dtype: int64
label
spam    51.05
ham     48.95
Name: proportion, dtype: float64

VALIDATION
label
spam    1710
ham     1648
Name: count, dtype: int64
label
spam    50.92
ham     49.08
Name: proportion, dtype: float64

TEST
label
spam    3382
ham     3313
Name: count, dtype: int64
label
spam    50.52
ham     49.48
Name: proportion, dtype: float64


In [22]:
train_groups = set(train_df["group_id"])
val_groups = set(val_df["group_id"])
test_groups = set(test_df["group_id"])

print(
    "Train ∩ Validation:",
    len(train_groups & val_groups)
)

print(
    "Train ∩ Test:",
    len(train_groups & test_groups)
)

print(
    "Validation ∩ Test:",
    len(val_groups & test_groups)
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [23]:
split_columns = [
    "message_id",
    "label",
    "target",
    "group_id",
    "split"
]

df[split_columns].to_csv(
    PROCESSED_DATA / "data_split.csv",
    index=False
)

print("Saved data_split.csv")

Saved data_split.csv
